In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.4 Temperature and decoding

The distribution is fixed; **decoding** is how we turn it into text.
Temperature reshapes the distribution before we sample from it.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
for t in [0, 0.5, 1.0, 1.5]:
    print(f"--- temperature={t} ---")
    print(generate(model, tok, "The ", max_new_tokens=60, temperature=t, seed=0))

Low temperature is confident and repetitive; high temperature is diverse
and chaotic. `top_k` and `top_p` instead clip the unlikely tail before sampling:

In [ ]:
print("top_k=5: ", generate(model, tok, "The ", max_new_tokens=60, top_k=5, seed=0))
print("top_p=0.9:", generate(model, tok, "The ", max_new_tokens=60, top_p=0.9, seed=0))

In [ ]:
# Temperature 0 is greedy argmax, so it is fully deterministic.
a = generate(model, tok, "The ", max_new_tokens=30, temperature=0)
b = generate(model, tok, "The ", max_new_tokens=30, temperature=0)
assert a == b